## Import packages

See YAML file for specific package requirements

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import tifffile
from shapely.geometry import Polygon
from shapely import wkt
import matplotlib.pyplot as plt
%matplotlib qt
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
import cv2
import rasterio
from rasterio.features import rasterize
from tensorflow.keras.preprocessing.image import load_img
from keras.saving import load_model
import segmenteverygrain as seg
import sez
from importlib import reload
from segment_anything import sam_model_registry, SamPredictor
from skimage.measure import regionprops, regionprops_table
from tqdm import trange, tqdm

## Setting up and loading models/checkpoints

Insert the path to your model, model checkpoints, and the image you would like to segment. The SAM model checkpoints can be downloaded from : https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

In [2]:
#path to where your segmenteveryzircon model is stored
model_fname = "/Users/omw339/Library/CloudStorage/OneDrive-TheUniversityofTexasatAustin/Desktop/segmenteverygrain/segmenteverygrain/zircons_2_26.keras"
# SAM model checkpoints
checkpoint = "/Users/omw339/Downloads/sam_vit_h_4b8939.pth"
#path of the image you would like to segment
image_fname = '/Users/omw339/Downloads/21-CON-17_EXP5_PPL.png'

load model and SAM checkpoints

In [3]:
model = load_model(model_fname, custom_objects={'weighted_crossentropy': seg.weighted_crossentropy})
sam = sam_model_registry["default"](checkpoint=checkpoint)

## Run segmentation

The following cell runs the segmentation model on the specified image to produce an inital output of grain geometries. 

For high resolution, large, images, this process may take a while. It may be necessary to [downsample](https://visionbook.mit.edu/upsamplig_downsampling_2.html) your image depending on the size of the image and the computing resources available to you.

In [4]:
all_grains, image_pred, all_coords = seg.predict_large_image(image_fname, model, sam, min_area=400.0, patch_size=2000, overlap=200, remove_large_objects=True)

segmenting image tiles...


100%|██████████| 8/8 [00:11<00:00,  1.41s/it]


creating masks using SAM...


100%|██████████| 24/24 [00:03<00:00,  6.64it/s]


finding overlapping polygons...


...

creating labeled image...
processed patch #5 out of 12 patches
segmenting image tiles...


100%|██████████| 2/2 [00:02<00:00,  1.46s/it]


processed patch #6 out of 12 patches
segmenting image tiles...


100%|██████████| 8/8 [00:12<00:00,  1.52s/it]


KeyboardInterrupt: 

## Plot initial prediction image and Initialize grain list/labels

In [8]:
image = np.array(load_img(image_fname))
fig, ax = plt.subplots()
all_grains, labels, pred_mask = seg.get_grains_from_patches(ax, image)
sez.plot_grains(image_fname, all_grains, step='Initial')

0it [00:00, ?it/s]
0it [00:00, ?it/s]
0it [00:00, ?it/s]


✅ Figure saved to: /Users/omw339/Downloads/21-CON-17_EXP5_PPL_Initialoutput.png


In [12]:
image = np.array(load_img(image_fname))
fig, ax = plt.subplots(figsize=(15,10))
plt.xticks([])
plt.yticks([])
seg.plot_image_w_colorful_grains(image, all_grains, ax, cmap='Paired')
seg.plot_grain_axes_and_centroids(all_grains, labels, ax, linewidth=1, markersize=10)
plt.axis('equal')
plt.xlim([0, np.shape(image)[1]])
plt.ylim([np.shape(image)[0], 0]);

0it [00:00, ?it/s]


Initial number of segmented polygons

In [12]:
initial_n = len(all_grains)
print(initial_n)

0


## Delete or merge grains in segmentation result
* click on the grain that you want to remove and press the 'x' key
* click on two grains that you want to merge and press the 'm' key (they have to be the last two grains you clicked on)
* press the 'g' key to hide the grain masks (so that you can see the original image better); press the 'g' key again to show the grain masks

In [ ]:
grain_inds = []
cid1 = fig.canvas.mpl_connect('button_press_event', 
                              lambda event: seg.onclick2(event, all_grains, grain_inds, ax=ax))
cid2 = fig.canvas.mpl_connect('key_press_event', 
                              lambda event: seg.onpress2(event, all_grains, grain_inds, fig=fig, ax=ax))

In [ ]:
fig.canvas.mpl_disconnect(cid1)
fig.canvas.mpl_disconnect(cid2)

Use this function to update the 'labels' array after deleting and merging grains (the 'all_grains' list is updated when doing the deletion and merging):

In [ ]:
all_grains, labels, gt_mask = seg.get_grains_from_patches(ax, image)

plot image after initial deletions

In [14]:
image = np.array(load_img(image_fname))
fig, ax = plt.subplots()
sez.plot_grains(image_fname, all_grains, step='Deletions')

0it [00:00, ?it/s]


✅ Figure saved to: /Users/omw339/Downloads/21-CON-17_EXP5_PPL_Initialoutput.png


## Add new grains using the Segment Anything Model

* click on unsegmented grain that you want to add
* press the 'x' key if you want to delete the last grain you added
* press the 'm' key if you want to merge the last two grains that you added
* right click outside the grain (but inside the most recent mask) if you want to restrict the grain to a smaller mask - this adds a background prompt

In [13]:
reload(seg)
predictor = SamPredictor(sam)
predictor.set_image(image) # this can take a while
coords = []
cid3 = fig.canvas.mpl_connect('button_press_event', lambda event: seg.onclick_large_image(event, ax, coords, image, predictor, patch_size=500))
cid4 = fig.canvas.mpl_connect('key_press_event', lambda event: seg.onpress(event, ax, fig))

In [14]:
fig.canvas.mpl_disconnect(cid3)
fig.canvas.mpl_disconnect(cid4)

In [15]:
all_grains, labels, gt_mask = seg.get_grains_from_patches(ax, image)

100%|██████████| 1/1 [00:00<00:00, 213.54it/s]
1it [00:00, 378.07it/s]


Plotting after additions and saving figure

In [ ]:
sez.plot_grains(image_fname, all_grains, step='Additions')

## Once you are happy with your segmentation results:

In [27]:
all_grains, labels, gt_mask = seg.get_grains_from_patches(ax, image)

100%|██████████| 21/21 [00:00<00:00, 3270.96it/s]
21it [00:00, 2397.27it/s]


In [16]:
final_n = len(all_grains)
print(final_n)

1


saving final figure of segmented polygons

In [ ]:
sez.plot_grains(image_fname, all_grains, step='Additions')

### Creating Metadata Table for Segmentation Results

In [17]:
df_meta = sez.create_metadata_table(image_fname, final_n, model_fname, save_csv=False)

After you are done with the deletion / addition of grain masks, run this cell to generate an updated set of grains:

## Last Steps: Save mask, grain polygons, and grain coordinates
- These steps are vital if you would like to come back to your work later 
- The below command creates the mask associated with the image you just segmented. The line should return 'True' meaning that the file was created. You can double check this by going into the folder where you saved the file.

### Saving binary mask 

In [ ]:
#saving binary mask
cv2.imwrite('/Users/omw339/Desktop/summer_SEZ_outputs/21-CON-17_EXP5_PPL_mask.png', gt_mask)

[ WARN:0@96869.256] global loadsave.cpp:848 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.


True

### Saving Grain Coordinates 
### NOTE: Essential for extracting measurements
- Run the following cells to create a geopandas geodataframe to store all of the polygons with their respective geometries and coordinates

In [ ]:
gdf = sez.grains_to_geodataframe(image_fname, all_grains)
gdf.head()

saving the csv file with the grain coordinates

In [ ]:
gdf.to_csv('/Users/omw339/Desktop/summer_SEZ_outputs/OWBP25012/OWBP25012_final_10_15.csv')

You are now done creating the grain polygons and masks. Use the morphometrics.ipynb to create the morphology dataset

## Steps to Re-Read your segmented grains back after finishing segmentation

### Step 1: Load polygons into geodataframe and re-initialize all_grains

In [ ]:
# Replace 'your_polygons.csv' with your actual CSV filename
csv_path = '/Users/omw339/Desktop/summer_SEZ_outputs/OWBP25001/OWBP25001_draft_coordinates_9_16_25_final.csv'

gdf = sez.load_polygons("path/to/your/file.csv", crs="EPSG:4326")
all_grains = list(gdf.geometry)

plot the loaded grains on the image

In [ ]:
sez.plot_grains(image_fname, all_grains, step='Initial')

now that all_grains is re-initialized and your grains are re-plotted, you are good to go back to adding and deleting grains!